# Distributional Regression Tutorial - DeepTab v2.0

This notebook demonstrates distributional regression (LSS models) for uncertainty quantification.

**What is distributional regression?**
Instead of predicting a single value, LSS models predict full probability distributions, enabling:
- Prediction intervals (e.g., 90% confidence)
- Quantile predictions (e.g., median, 5th percentile)
- Uncertainty quantification
- Handling asymmetric distributions

**Topics covered:**
- Training LSS models with different distribution families
- Generating prediction intervals
- Validating interval coverage
- Visualizing uncertainty

**Requirements:**
```bash
pip install deeptab scikit-learn pandas numpy matplotlib scipy
```

## 1. Setup and Data Generation

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split

from deeptab.models import MambularLSS

# Set random seed
np.random.seed(42)

# Generate synthetic data
n_samples, n_features = 1000, 5
X = np.random.randn(n_samples, n_features)
y = np.dot(X, np.random.randn(n_features)) + np.random.randn(n_samples)

# Create DataFrame
df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(n_features)])
df["target"] = y

print(f"Dataset shape: {df.shape}")
print(f"Target mean: {y.mean():.3f}, std: {y.std():.3f}")

## 2. Train/Test Split

In [ ]:
X = df.drop(columns=["target"])
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 3. Train LSS Model with Normal Distribution

For continuous symmetric targets, use the "normal" family.

In [ ]:
# Train LSS model
model = MambularLSS()
model.fit(X_train, y_train, family="normal", max_epochs=50)

## 4. Predict Distribution Parameters

In [ ]:
# Get distribution parameters
params = model.predict(X_test)
print(f"Parameters shape: {params.shape}")
print("Column 0: mean, Column 1: log(std)")

# Extract mean and std
mean = params[:, 0]
log_std = params[:, 1]
std = np.exp(log_std)

print(f"\nMean of predicted means: {mean.mean():.3f}")
print(f"Mean of predicted stds: {std.mean():.3f}")
print(f"\nSample parameters:")
for i in range(5):
    print(f"  Sample {i}: mean={mean[i]:.3f}, std={std[i]:.3f}")

## 5. Generate Prediction Intervals

Create prediction intervals at different confidence levels.

In [ ]:
# Generate intervals at multiple confidence levels
print("Prediction Interval Coverage:")
print("="*40)

for confidence in [0.50, 0.68, 0.90, 0.95]:
    alpha = 1 - confidence
    z = stats.norm.ppf(1 - alpha / 2)
    
    lower = mean - z * std
    upper = mean + z * std
    
    coverage = np.mean((y_test >= lower) & (y_test <= upper))
    print(f"{confidence*100:5.0f}% interval: empirical coverage = {coverage:.3f}")

## 6. Visualize Predictions with Uncertainty

Plot predictions with 90% prediction intervals.

In [ ]:
import matplotlib.pyplot as plt

# Plot first 50 samples
n_plot = 50
indices = np.arange(n_plot)

fig, ax = plt.subplots(figsize=(14, 6))

# Actual values
ax.scatter(indices, y_test[:n_plot], color="black", label="Actual", alpha=0.7, s=50)

# Predicted means
ax.scatter(indices, mean[:n_plot], color="blue", label="Predicted mean", alpha=0.7, s=50)

# 90% prediction intervals
z_90 = stats.norm.ppf(0.95)
lower_90 = mean[:n_plot] - z_90 * std[:n_plot]
upper_90 = mean[:n_plot] + z_90 * std[:n_plot]

ax.fill_between(indices, lower_90, upper_90, alpha=0.3, color="blue", label="90% interval")

ax.set_xlabel("Sample Index")
ax.set_ylabel("Target Value")
ax.set_title("LSS Predictions with 90% Uncertainty Intervals")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Statistics
in_interval = np.sum((y_test[:n_plot] >= lower_90) & (y_test[:n_plot] <= upper_90))
print(f"Points within 90% interval: {in_interval}/{n_plot} ({in_interval/n_plot:.1%})")

## 7. Visualize Individual Distributions

Plot predicted distributions for specific samples.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i in range(6):
    mean_i = mean[i]
    std_i = std[i]
    actual_i = y_test[i]
    
    # Create distribution
    x_range = np.linspace(mean_i - 4*std_i, mean_i + 4*std_i, 100)
    pdf = stats.norm.pdf(x_range, loc=mean_i, scale=std_i)
    
    # Plot
    axes[i].plot(x_range, pdf, 'b-', linewidth=2, label='Predicted dist')
    axes[i].axvline(actual_i, color='red', linestyle='--', linewidth=2, label='Actual')
    axes[i].axvline(mean_i, color='blue', linestyle='--', linewidth=2, alpha=0.5, label='Mean')
    axes[i].set_title(f'Sample {i}: μ={mean_i:.2f}, σ={std_i:.2f}')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    if i == 0:
        axes[i].legend()

plt.tight_layout()
plt.show()

## 8. Coverage Validation (Calibration Plot)

Check if prediction intervals are well-calibrated across confidence levels.

In [ ]:
# Test multiple confidence levels
confidence_levels = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

results = []
for confidence in confidence_levels:
    alpha = 1 - confidence
    z = stats.norm.ppf(1 - alpha / 2)
    
    lower = mean - z * std
    upper = mean + z * std
    
    coverage = np.mean((y_test >= lower) & (y_test <= upper))
    results.append((confidence, coverage))

# Plot calibration
plt.figure(figsize=(8, 8))
plt.plot([r[0] for r in results], [r[1] for r in results], 'o-', linewidth=2, markersize=8, label='Observed')
plt.plot([0.5, 1.0], [0.5, 1.0], 'r--', linewidth=2, label='Perfect calibration')
plt.xlabel('Nominal Coverage', fontsize=12)
plt.ylabel('Empirical Coverage', fontsize=12)
plt.title('Prediction Interval Calibration', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim(0.45, 1.0)
plt.ylim(0.45, 1.0)
plt.tight_layout()
plt.show()

print("Calibration results:")
for conf, cov in results:
    diff = abs(cov - conf)
    status = "✓" if diff < 0.05 else "⚠"
    print(f"  {status} {conf:.0%} nominal → {cov:.1%} empirical (diff: {diff:.1%})")

## 9. Quantile Predictions

Extract specific quantiles from the predicted distributions.

In [ ]:
# Get various quantiles
q05 = stats.norm.ppf(0.05, loc=mean, scale=std)
q25 = stats.norm.ppf(0.25, loc=mean, scale=std)
q50 = stats.norm.ppf(0.50, loc=mean, scale=std)  # median
q75 = stats.norm.ppf(0.75, loc=mean, scale=std)
q95 = stats.norm.ppf(0.95, loc=mean, scale=std)

print("Sample Quantile Predictions:")
print("="*60)
for i in range(10):
    print(f"Sample {i}: actual={y_test[i]:6.3f}, "
          f"P5={q05[i]:6.3f}, P25={q25[i]:6.3f}, P50={q50[i]:6.3f}, "
          f"P75={q75[i]:6.3f}, P95={q95[i]:6.3f}")

## 10. Compare with Point-Estimate Regressor

In [ ]:
from deeptab.models import MambularRegressor

# Train point-estimate regressor
regressor = MambularRegressor()
regressor.fit(X_train, y_train, max_epochs=50)

# Compare predictions
point_pred = regressor.predict(X_test)
lss_mean = mean

from sklearn.metrics import mean_squared_error, r2_score

rmse_point = np.sqrt(mean_squared_error(y_test, point_pred))
rmse_lss = np.sqrt(mean_squared_error(y_test, lss_mean))

r2_point = r2_score(y_test, point_pred)
r2_lss = r2_score(y_test, lss_mean)

print("Point Regressor vs LSS Model:")
print("="*40)
print(f"Point regressor - RMSE: {rmse_point:.3f}, R²: {r2_point:.3f}")
print(f"LSS mean       - RMSE: {rmse_lss:.3f}, R²: {r2_lss:.3f}")
print(f"\nLSS advantage: Provides uncertainty estimates!")
print(f"Mean predicted std: {std.mean():.3f}")

## 11. Using Different Distribution Families

Try different families for different data types.

In [ ]:
# For positive data, use Gamma distribution
y_positive = np.abs(y) + 1.0

X_train_pos, X_test_pos, y_train_pos, y_test_pos = train_test_split(
    X, y_positive, test_size=0.2, random_state=42
)

# Train with Gamma family
model_gamma = MambularLSS()
model_gamma.fit(X_train_pos, y_train_pos, family="gamma", max_epochs=50)

# Get parameters
params_gamma = model_gamma.predict(X_test_pos)
log_alpha = params_gamma[:, 0]  # Shape
log_beta = params_gamma[:, 1]   # Rate

alpha = np.exp(log_alpha)
beta = np.exp(log_beta)

mean_gamma = alpha / beta
variance_gamma = alpha / (beta ** 2)

print("Gamma Distribution Results:")
print("="*40)
print(f"Actual mean: {y_test_pos.mean():.3f}")
print(f"Predicted mean: {mean_gamma.mean():.3f}")
print(f"\nSample parameters:")
for i in range(5):
    print(f"  Sample {i}: α={alpha[i]:.3f}, β={beta[i]:.3f}, "
          f"mean={mean_gamma[i]:.3f}")

## 12. Poisson for Count Data

In [ ]:
# Generate count data
X_count = np.random.randn(1000, 5)
lambda_true = np.exp(X_count @ np.random.randn(5) * 0.5)
y_count = np.random.poisson(lambda_true)

X_train_count, X_test_count, y_train_count, y_test_count = train_test_split(
    X_count, y_count, test_size=0.2, random_state=42
)

# Train with Poisson family
model_poisson = MambularLSS()
model_poisson.fit(X_train_count, y_train_count, family="poisson", max_epochs=50)

# Get rate parameter
params_poisson = model_poisson.predict(X_test_count)
log_lambda = params_poisson[:, 0]
lambda_pred = np.exp(log_lambda)

print("Poisson Distribution Results:")
print("="*40)
print(f"Actual mean: {y_test_count.mean():.3f}")
print(f"Predicted mean (λ): {lambda_pred.mean():.3f}")
print(f"\nFor Poisson: mean = variance = λ")
print(f"Sample λ values: {lambda_pred[:10]}")

## 13. Available Distribution Families

In [ ]:
families_info = {
    "normal": "Continuous symmetric (unbounded)",
    "poisson": "Non-negative integer counts",
    "gamma": "Strictly positive continuous (right skew)",
    "beta": "Bounded to [0, 1] interval",
    "negative_binomial": "Overdispersed count data",
    "student_t": "Continuous with heavy tails (robust to outliers)",
}

print("Available Distribution Families:")
print("="*60)
for family, description in families_info.items():
    print(f"  {family:20s} - {description}")

print("\n✓ Choose family based on target characteristics")
print("✓ All families supported by all LSS models")

## 14. Evaluate with Negative Log-Likelihood

In [ ]:
# Evaluate LSS model
metrics = model.evaluate(X_test, y_test)
print(f"Negative Log-Likelihood: {metrics['loss']:.3f}")
print("(Lower is better)")

## 15. Save and Load

In [ ]:
# Save model
model.save("lss_model.pkl")
print("LSS model saved!")

# Load model
loaded_model = MambularLSS.load("lss_model.pkl")

# Use loaded model
params_loaded = loaded_model.predict(X_test)
print(f"Loaded model parameters shape: {params_loaded.shape}")

## Summary

In this tutorial, you learned how to:
- ✅ Train LSS models for distributional regression
- ✅ Predict full probability distributions (not just point estimates)
- ✅ Generate prediction intervals at any confidence level
- ✅ Validate interval calibration
- ✅ Visualize uncertainty with plots
- ✅ Use different distribution families (normal, gamma, poisson)
- ✅ Extract quantiles from predicted distributions
- ✅ Compare LSS with point-estimate regressors

**Key advantages of LSS models:**
- Uncertainty quantification
- Asymmetric prediction intervals
- Handles different target distributions
- Better decision-making under uncertainty

**Next steps:**
- Try [Classification Tutorial](classification.ipynb) for categorical targets
- Check [Regression Tutorial](regression.ipynb) for point estimates
- Explore [Experimental Models](experimental.ipynb) for cutting-edge architectures

**Documentation:** https://deeptab.readthedocs.io/